# Dataset Corruption & Faithfulness Evaluation Pipeline

**Task 1 — Corruption** using `jhu-clsp/mmBERT-base` (MLM encoder)  
**Task 2 — Evaluation** using `KRLabsOrg/lettucedect-large-modernbert-en-v1`

### Flow
```
dataset_v3_tagged.json
        │
        ▼
  [Task 1] mmBERT masks JSON string values
           → if top-1 == original: replace with top-2
           → skip numbers / dates / percentages
        │
        ▼
 dataset_v3_corrupted.json
        │
        ▼
  [Task 2] lettucedect checks faithfulness
           question = user_prompt
           answer   = original_answer
           context  = tool_response (JSON → flat text)
        │
        ▼
 dataset_v3_evaluated.json
```

## 0 · Install dependencies

In [ ]:
!pip install -q transformers torch accelerate tqdm lettuce-detect

## 1 · Imports & global config

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # the largest/fastest GPU

In [2]:
import json
import re
from copy import deepcopy
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE     = "dataset_v3_tagged.json"
CORRUPTED_FILE = "dataset_v3_corrupted.json"
EVALUATED_FILE = "dataset_v3_evaluated.json"

# ── Models ─────────────────────────────────────────────────────────────────
MLM_MODEL_NAME     = "jhu-clsp/mmBERT-base"                           # Task 1
LETTUCE_MODEL_NAME = "KRLabsOrg/lettucedect-large-modernbert-en-v1"   # Task 2

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


/mnt/raid/venskii/tool_calling/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 2 · Load & inspect dataset

In [ ]:
with open(INPUT_FILE, encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Total rows : {len(dataset)}")
print(f"Keys       : {list(dataset[0].keys())}")
print()

# Preview first 2 rows
for i, row in enumerate(dataset[:2]):
    print(f"{'='*60}")
    print(f"ROW {i}")
    print(f"{'='*60}")
    print(f"[user_prompt]\n{row['user_prompt']}\n")
    print(f"[tool_call]\n{row['tool_call']}\n")
    tr_preview = json.loads(row['tool_response'])
    print(f"[tool_response (parsed)]\n{json.dumps(tr_preview, indent=2)[:400]}...\n")
    print(f"[tagged_answer]\n{row['tagged_answer'][:300]}...\n")

---
## 3 · Task 1 — Corruption with mmBERT

### 3.1 · Load mmBERT model

In [ ]:
print(f"Loading MLM model: {MLM_MODEL_NAME} ...")
mlm_tokenizer = AutoTokenizer.from_pretrained(MLM_MODEL_NAME)
mlm_model     = AutoModelForMaskedLM.from_pretrained(MLM_MODEL_NAME).to(DEVICE)
mlm_model.eval()
MASK_TOKEN = mlm_tokenizer.mask_token
print(f"Mask token : '{MASK_TOKEN}'")
print("Model loaded.")

### 3.2 · Helper functions

In [ ]:
# Patterns for values that must NEVER be corrupted
SKIP_PATTERNS = [
    r"^-?\d+(\.\d+)?$",           # plain numbers:   4172.80, -3, 0.5
    r"^[+-]?\d+(\.\d+)?%$",       # percentages:     +0.68%, -1.2%
    r"^\d{4}-\d{2}-\d{2}",        # ISO dates:       2024-01-01
    r"^\d{1,2}/\d{1,2}/\d{2,4}",  # US dates:        01/15/2024
    r"^-?\d{1,3}\.\d+$",          # coordinates:     37.7749, -122.4194
]


def should_skip(value: str) -> bool:
    """Return True for numeric / date / percentage values — never corrupt these."""
    v = value.strip()
    return any(re.match(p, v) for p in SKIP_PATTERNS)


def extract_leaf_strings(obj) -> list[str]:
    """
    Recursively collect all leaf string values from a JSON object
    that are non-empty and not numeric/date.
    """
    results = []
    if isinstance(obj, dict):
        for v in obj.values():
            if isinstance(v, str) and v.strip() and not should_skip(v):
                results.append(v)
            elif isinstance(v, (dict, list)):
                results.extend(extract_leaf_strings(v))
    elif isinstance(obj, list):
        for item in obj:
            results.extend(extract_leaf_strings(item))
    return results


def get_mlm_top_k(context: str, value: str, top_k: int = 5) -> list[tuple[str, float]]:
    """
    Mask the first occurrence of `value` in `context`, run mmBERT,
    return top-k (token, probability) pairs.
    """
    masked = context.replace(value, MASK_TOKEN, 1)
    if MASK_TOKEN not in masked:
        return []

    inputs = mlm_tokenizer(
        masked,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)

    mask_ids = (inputs["input_ids"] == mlm_tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    if len(mask_ids) == 0:
        return []

    with torch.no_grad():
        logits = mlm_model(**inputs).logits  # (1, seq_len, vocab)

    probs = torch.softmax(logits[0, mask_ids[0]], dim=-1)
    top   = torch.topk(probs, k=top_k)

    return [
        (mlm_tokenizer.decode([idx]).strip(), score)
        for score, idx in zip(top.values.tolist(), top.indices.tolist())
    ]


def maybe_corrupt(value: str, context: str) -> Optional[str]:
    """
    Try to corrupt a single string value using mmBERT MLM predictions.
    - top-1 == original  →  return top-2 token  (confident prediction → corrupt)
    - top-1 != original  →  return None          (uncertain → leave unchanged)
    """
    preds = get_mlm_top_k(context, value)
    if len(preds) < 2:
        return None

    top1, top2 = preds[0][0], preds[1][0]
    if top1.lower() == value.lower().strip():
        # Model was right → use second-best as the corruption
        return top2 if top2.lower() != value.lower() else None
    return None


def corrupt_tool_response(tool_response_str: str, tagged_answer: str) -> tuple[str, dict]:
    """
    Walk the parsed tool_response JSON, corrupt eligible string values,
    and return the modified JSON string + a log of changes.
    """
    try:
        tr = json.loads(tool_response_str)
    except (json.JSONDecodeError, TypeError):
        return tool_response_str, {}

    eligible = set(extract_leaf_strings(tr))
    # Rich context: tagged answer + beginning of the raw JSON
    context = tagged_answer + "\n\n" + tool_response_str[:1000]
    log = {}

    def _walk(obj):
        if isinstance(obj, dict):
            return {k: _walk(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_walk(item) for item in obj]
        if isinstance(obj, str) and obj in eligible:
            corrupted = maybe_corrupt(obj, context)
            if corrupted:
                log[obj] = corrupted
                return corrupted
        return obj  # numbers / booleans / None / skipped strings untouched

    corrupted_tr = _walk(tr)
    return json.dumps(corrupted_tr, ensure_ascii=False), log

### 3.3 · Run corruption over full dataset

In [ ]:
corrupted_dataset = []
total_corruptions = 0

for row in tqdm(dataset, desc="Corrupting", unit="row"):
    new_row = deepcopy(row)

    context_text = row.get("tagged_answer") or row.get("original_answer", "")
    corrupted_tr, log = corrupt_tool_response(row["tool_response"], context_text)

    new_row["tool_response"]  = corrupted_tr
    new_row["corruption_log"] = log
    total_corruptions += len(log)
    corrupted_dataset.append(new_row)

print(f"\nRows processed      : {len(corrupted_dataset)}")
print(f"Total value changes : {total_corruptions}")
print(f"Avg changes / row   : {total_corruptions / len(corrupted_dataset):.2f}")

### 3.4 · Inspect a few corruptions

In [ ]:
examples_shown = 0
for row in corrupted_dataset:
    if row["corruption_log"] and examples_shown < 5:
        print(f"user_prompt : {row['user_prompt'][:80]}...")
        for orig, corrupt in row["corruption_log"].items():
            print(f"  '{orig}'  →  '{corrupt}'")
        print()
        examples_shown += 1

### 3.5 · Save corrupted dataset

In [ ]:
with open(CORRUPTED_FILE, "w", encoding="utf-8") as f:
    json.dump(corrupted_dataset, f, ensure_ascii=False, indent=2)

print(f"Saved → {CORRUPTED_FILE}  ({Path(CORRUPTED_FILE).stat().st_size / 1024:.1f} KB)")

---
## 4 · Task 2 — Faithfulness evaluation with lettucedect

### 4.1 · Load lettucedect model

In [ ]:
from transformers import pipeline as hf_pipeline

print(f"Loading NLI model: {LETTUCE_MODEL_NAME} ...")
# lettucedect is an NLI sequence-classification model
nli_pipe = hf_pipeline(
    task="text-classification",
    model=LETTUCE_MODEL_NAME,
    device=0 if DEVICE == "cuda" else -1,
    truncation=True,
    max_length=512,
)
print("Model loaded.")

### 4.2 · Helper functions

In [ ]:
def flatten_json_to_text(json_str: str) -> str:
    """
    Convert a tool_response JSON string into clean flat text.
    NLI models work better on prose than raw JSON syntax.
    """
    try:
        obj = json.loads(json_str)
    except (json.JSONDecodeError, TypeError):
        return str(json_str)

    lines = []

    def _walk(node, depth=0):
        pad = "  " * depth
        if isinstance(node, dict):
            for k, v in node.items():
                if isinstance(v, (dict, list)):
                    lines.append(f"{pad}{k}:")
                    _walk(v, depth + 1)
                else:
                    lines.append(f"{pad}{k}: {v}")
        elif isinstance(node, list):
            for item in node:
                _walk(item, depth)
        else:
            lines.append(f"{pad}{node}")

    _walk(obj)
    return "\n".join(lines)


def split_sentences(text: str) -> list[str]:
    """Naive sentence splitter — good enough for answer sentences."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sentences if s.strip()]


def evaluate_faithfulness(question: str, answer: str, context: str) -> dict:
    """
    Check each sentence of `answer` against `context` using lettucedect.
    Returns:
        {
          "sentences": [{"sentence": ..., "label": ..., "score": ...}, ...],
          "faithfulness_score": float,   # fraction of supported sentences
          "label_counts": {"supported": N, "unsupported": N, "contradicted": N}
        }
    """
    sentences = split_sentences(answer)
    if not sentences:
        return {"sentences": [], "faithfulness_score": None, "label_counts": {}}

    # lettucedect expects: premise = context, hypothesis = sentence
    # We prepend the question for additional grounding
    premise = f"Question: {question}\n\nContext:\n{context}"

    pairs = [{"text": premise, "text_pair": s} for s in sentences]
    raw_preds = nli_pipe(pairs)

    results = []
    for sent, pred in zip(sentences, raw_preds):
        results.append({
            "sentence": sent,
            "label":    pred["label"].lower(),   # supported / unsupported / contradicted
            "score":    round(pred["score"], 4),
        })

    labels = [r["label"] for r in results]
    n_supported = labels.count("supported")

    return {
        "sentences":         results,
        "faithfulness_score": round(n_supported / len(labels), 4),
        "label_counts": {
            "supported":    n_supported,
            "unsupported":  labels.count("unsupported"),
            "contradicted": labels.count("contradicted"),
        },
    }

### 4.3 · Run evaluation over corrupted dataset

In [ ]:
# Load the corrupted dataset (or swap in INPUT_FILE to evaluate the originals)
with open(CORRUPTED_FILE, encoding="utf-8") as f:
    eval_source = json.load(f)

evaluated_dataset = []

for row in tqdm(eval_source, desc="Evaluating", unit="row"):
    new_row = deepcopy(row)

    context = flatten_json_to_text(row["tool_response"])

    eval_result = evaluate_faithfulness(
        question=row["user_prompt"],
        answer=row["original_answer"],
        context=context,
    )

    new_row["faithfulness_eval"]  = eval_result["sentences"]
    new_row["faithfulness_score"] = eval_result["faithfulness_score"]
    new_row["label_counts"]       = eval_result["label_counts"]
    evaluated_dataset.append(new_row)

print(f"Evaluation complete — {len(evaluated_dataset)} rows")

### 4.4 · Summary statistics

In [ ]:
scores = [r["faithfulness_score"] for r in evaluated_dataset if r["faithfulness_score"] is not None]

avg      = sum(scores) / len(scores)
perfect  = sum(1 for s in scores if s == 1.0)
low      = sum(1 for s in scores if s < 0.5)

print(f"{'='*50}")
print(f" FAITHFULNESS SUMMARY  ({len(scores)} rows)")
print(f"{'='*50}")
print(f"  Average score        : {avg:.4f}")
print(f"  Fully faithful (1.0) : {perfect}")
print(f"  Low score  (< 0.50)  : {low}")
print(f"  Min score            : {min(scores):.4f}")
print(f"  Max score            : {max(scores):.4f}")

# Aggregate label counts across all rows
total_s = sum(r["label_counts"].get("supported",    0) for r in evaluated_dataset)
total_u = sum(r["label_counts"].get("unsupported",  0) for r in evaluated_dataset)
total_c = sum(r["label_counts"].get("contradicted", 0) for r in evaluated_dataset)
grand   = total_s + total_u + total_c

print()
print(f"  Sentence label breakdown ({grand} total sentences):")
print(f"    supported    : {total_s}  ({100*total_s/grand:.1f}%)")
print(f"    unsupported  : {total_u}  ({100*total_u/grand:.1f}%)")
print(f"    contradicted : {total_c}  ({100*total_c/grand:.1f}%)")

### 4.5 · Inspect low-faithfulness examples

In [ ]:
low_faith = sorted(
    [r for r in evaluated_dataset if r["faithfulness_score"] is not None],
    key=lambda r: r["faithfulness_score"]
)[:3]

for row in low_faith:
    print(f"{'─'*60}")
    print(f"Score : {row['faithfulness_score']}")
    print(f"Q     : {row['user_prompt'][:100]}")
    print(f"Corruptions: {row.get('corruption_log', {})}")
    print("Sentence labels:")
    for s in row["faithfulness_eval"]:
        print(f"  [{s['label']:>12s}  {s['score']:.3f}]  {s['sentence'][:100]}")
    print()

### 4.6 · Save evaluated dataset

In [ ]:
with open(EVALUATED_FILE, "w", encoding="utf-8") as f:
    json.dump(evaluated_dataset, f, ensure_ascii=False, indent=2)

print(f"Saved → {EVALUATED_FILE}  ({Path(EVALUATED_FILE).stat().st_size / 1024:.1f} KB)")